# Cat Embeddings Model
ShuffleNetV2 x0.5 repurposed to create a lightweight embeddings model for cat identity and behaviour classification

## To do
- ✅ Remove classification layers
- ✅ Add ONNX export path
- ✅ Demo embeddings using clustering

## Configure notebook

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from pathlib import Path

import numpy as np
import torch
from matplotlib import pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans
from torchvision import models

import utils
from config import settings

In [ ]:
%matplotlib inline

In [ ]:
device = utils.get_best_device()

## Load and prepare embeddings model

In [ ]:
# Load backbone model
weights = models.ShuffleNet_V2_X0_5_Weights.DEFAULT
preprocess = models.ShuffleNet_V2_X0_5_Weights.DEFAULT.transforms(
    crop_size=settings.EMBEDDING_IMGSZ, resize_size=settings.EMBEDDING_IMGSZ
)
backbone = models.shufflenet_v2_x0_5(weights=weights)
embedding_model = torch.nn.Sequential(
    backbone.conv1,
    backbone.maxpool,
    backbone.stage2,
    backbone.stage3,
    backbone.stage4,
    backbone.conv5,
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(1),
)
_ = embedding_model.to(device).eval()

## Quantise model

In [ ]:
onnx_path = (
    Path("..") / "models_staging" / "shufflenetv2_x0_5_embeddings_onnx_model.onnx"
)
quantized_model = embedding_model.to("cpu").half().eval()
dummy = torch.randn(
    1, 3, settings.EMBEDDING_IMGSZ, settings.EMBEDDING_IMGSZ, dtype=torch.float16
)
onnx_program = torch.onnx.export(
    quantized_model,
    (dummy,),
    input_names=["images"],
    output_names=["embeddings"],
    opset_version=18,
    dynamic_shapes=None,
)
onnx_program.save(onnx_path, external_data=False)

## Cluster sample images by embedding similarity

In [ ]:
# Create clusters
import tracking

sample_images = sorted((Path("datasets") / "finetune_data").rglob("*.jpg"))
embeddings = np.stack(
    [
        tracking.embed_image(np.asarray(Image.open(path).convert("RGB")))
        for path in sample_images
    ]
)

n_clusters = min(3, len(sample_images))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
labels = kmeans.fit_predict(embeddings)

clusters = {cluster_id: [] for cluster_id in range(n_clusters)}
for path, label in zip(sample_images, labels):
    clusters[label].append(path)

In [ ]:
# Display clusters
max_cols = max(len(paths) for paths in clusters.values())
fig, axes = plt.subplots(
    n_clusters, max_cols, figsize=(3 * max_cols, 3 * n_clusters), squeeze=False
)

for row in range(n_clusters):
    paths = clusters[row]
    for col in range(max_cols):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(paths):
            image = np.asarray(Image.open(paths[col]).convert("RGB"))
            ax.imshow(image)
            ax.set_title(f"C{row}: {paths[col].name}", fontsize=9)

fig.tight_layout()